# Straylight

### Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
import scipy.integrate as integrate

In [ ]:
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time

In [ ]:
import h5py

In [ ]:
from matplotlib import pyplot as plt
from matplotlib.ticker import AutoMinorLocator, MultipleLocator

In [ ]:
import colorcet as cc

In [ ]:
import plotly.express as px
import plotly.io as pio

### Plot config

In [ ]:
plt.rc('font',   size=12)          # controls default text sizes
plt.rc('axes',   titlesize=12)     # fontsize of the axes title
plt.rc('axes',   labelsize=12)     # fontsize of the x and y labels
plt.rc('xtick',  labelsize=12)     # fontsize of the tick labels
plt.rc('ytick',  labelsize=12)     # fontsize of the tick labels
plt.rc('legend', fontsize=12)      # legend fontsize
plt.rc('figure', titlesize=12)     # fontsize of the figure title

### Read the orbit file

In [ ]:
df = pd.read_csv("../../../inputfiles/Plato_straylight_example_index_2574.csv")

In [ ]:
df.columns

Add a julian day time column, which will make things easier to plot. The astropy library expects time stamps of the format `2026-06-11T19:00:26` rather than `20260611T190026` so we need to reformat them first.

In [ ]:
times = [t[:4] + "-" + t[4:6] + "-" + t[6:11] + ":" + t[11:13] + ":" + t[13:] for t in df['#Date'].values]
times = Time(times, format='isot', scale='utc')
df['jd'] = times.jd                                  # Julian day

## Typical distance of Plato to the Moon

In [ ]:
df["dist_sc_moon"] = np.sqrt((df['X_SC_EME2000 [km]'] -  df['X_MOON_EME2000 [km]'])**2.     \
                             + (df['Y_SC_EME2000 [km]'] -  df['Y_MOON_EME2000 [km]'])**2.   \
                             + (df['Z_SC_EME2000 [km]'] -  df['Z_MOON_EME2000 [km]'])**2.)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df.loc[250:, 'jd'] - 2400000, df.loc[250:, "dist_sc_moon"])
ax.set_xlabel("Julian Day - 2400000 [d]")
ax.set_ylabel("Distance Moon-Plato [km]")
ax.set_xlim(61300, 62000)
plt.tight_layout()

## Typical distance between the Moon and the Sun

In [ ]:
df["dist_sun_moon"] = np.sqrt((df['X_SUN_EME2000 [km]'] -  df['X_MOON_EME2000 [km]'])**2.     \
                            + (df['Y_SUN_EME2000 [km]'] -  df['Y_MOON_EME2000 [km]'])**2.   \
                            + (df['Z_SUN_EME2000 [km]'] -  df['Z_MOON_EME2000 [km]'])**2.)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df.loc[250:, 'jd'] - 2400000, df.loc[250:, "dist_sun_moon"])
ax.set_xlabel("Julian day - 2400000 [d]")
ax.set_ylabel("Distance Moon-Sun [km]")
ax.set_xlim(61300, 62000)
plt.tight_layout()

## The apparent size of the Moon seen from Plato spacecraft

In [ ]:
radius_moon = 1737.4                      # [km]
diameter_moon = 2 * radius_moon           # [km]

In [ ]:
apparent_diameter = diameter_moon / df["dist_sc_moon"] / np.pi * 180.                            # [deg]

Skip the first 249 days as they include the journey to L2.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df.loc[250:2000-1, 'jd'] - 2400000, apparent_diameter[250:2000])
ax.set_xlabel("Julian day - 2400000 [d]")
ax.set_ylabel("Apparent diameter [deg]")
ax.set_xlim(61300, 62000)
plt.tight_layout()
plt.savefig("apparent_diameter_moon.png", dpi=150)

## Angle between Optical Axis and the direction to the Moon

In [ ]:
lops2 = SkyCoord('06h21m14.5s', '-47d53m13s', frame='icrs')
print("(RA, dec) = ", lops2.ra.deg, lops2.dec.deg, " deg")
print("(RA, dec) = ", lops2.ra.rad, lops2.dec.rad, " rad")

The unit vector pointing to the optical axis:

In [ ]:
ux_oa = np.cos(lops2.dec.rad) * np.cos(lops2.ra.rad)
uy_oa = np.cos(lops2.dec.rad) * np.sin(lops2.ra.rad)
uz_oa = np.sin(lops2.dec.rad)

In [ ]:
df["cosBeta"] = ux_oa * (df["X_MOON_EME2000 [km]"] - df["X_SC_EME2000 [km]"]) / df["dist_sc_moon"]    \
              + uy_oa * (df["Y_MOON_EME2000 [km]"] - df["Y_SC_EME2000 [km]"]) / df["dist_sc_moon"]    \
              + uz_oa * (df["Z_MOON_EME2000 [km]"] - df["Z_SC_EME2000 [km]"]) / df["dist_sc_moon"]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df.loc[250:2000, 'jd'] - 2400000, df.loc[250:2000, "cosBeta"])
ax.set_xlabel("Julian day - 2400000 [d]")
ax.set_ylabel("cos(angle LOPS2-Moon)")
ax.set_xlim(61300, 62500)
plt.grid()
plt.tight_layout()
plt.savefig("cosBeta.png", dpi=150)

In [ ]:
df.loc[250, "Date"]

## Angle between Optical Axis and the direction to the Sun

In [ ]:
df["dist_sun_plato"] = np.sqrt(  (df["X_SUN_EME2000 [km]"] - df["X_MOON_EME2000 [km]"])**2
                               + (df["Y_SUN_EME2000 [km]"] - df["Y_MOON_EME2000 [km]"])**2
                               + (df["Z_SUN_EME2000 [km]"] - df["Z_MOON_EME2000 [km]"])**2 ) 

In [ ]:
df["cosPsi"] = ux_oa * (df["X_SUN_EME2000 [km]"] - df["X_SC_EME2000 [km]"]) / df["dist_sun_plato"]    \
             + uy_oa * (df["Y_SUN_EME2000 [km]"] - df["Y_SC_EME2000 [km]"]) / df["dist_sun_plato"]    \
             + uz_oa * (df["Z_SUN_EME2000 [km]"] - df["Z_SC_EME2000 [km]"]) / df["dist_sun_plato"]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df.loc[250:2000, 'jd'] - 2400000, df.loc[250:2000, "cosPsi"])
ax.set_xlabel("Julian day - 2400000 [d]")
ax.set_ylabel("cos(angle LOPS2-Sun)")
ax.set_xlim(61300, 62500)
plt.tight_layout()
plt.savefig("cosPsi.png", dpi=150)

## Visualization of orbit and direction of Sun and Moon

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

begin = 250
end = 650
middle = begin + (end-begin)/2

ax.scatter([0], [0], s=40, c="blue")
ax.scatter(df.loc[begin, "X_SC_EME2000 [km]"]/1.e6, df.loc[begin, "Y_SC_EME2000 [km]"] / 1.e6, s=40, c="red")


ax.plot(df.loc[begin:end, "X_MOON_EME2000 [km]"]/1.e6, df.loc[begin:end, "Y_MOON_EME2000 [km]"] / 1.e6, c="gray", linewidth=0.1)
ax.plot(df.loc[begin:end, "X_SC_EME2000 [km]"]/1.e6, df.loc[begin:end, "Y_SC_EME2000 [km]"] / 1.e6, c="red", linewidth=1)

for n in range(begin,end,30):

    plt.arrow(df.loc[n, "X_SC_EME2000 [km]"]/1.e6, df.loc[n, "Y_SC_EME2000 [km]"] / 1.e6,           \
              (df.loc[n, "X_SUN_EME2000 [km]"]/1.e6 - df.loc[n, "X_SC_EME2000 [km]"]/1.e6)/1000,    \
              (df.loc[n, "Y_SUN_EME2000 [km]"]/1.e6 - df.loc[n, "Y_SC_EME2000 [km]"]/1.e6)/1000,    \
              color="orange", head_width=0.03)

    plt.arrow(df.loc[n, "X_SC_EME2000 [km]"]/1.e6, df.loc[n, "Y_SC_EME2000 [km]"] / 1.e6,           \
              (df.loc[n, "X_MOON_EME2000 [km]"]/1.e6 - df.loc[n, "X_SC_EME2000 [km]"]/1.e6)/10,    \
              (df.loc[n, "Y_MOON_EME2000 [km]"]/1.e6 - df.loc[n, "Y_SC_EME2000 [km]"]/1.e6)/10,    \
              color="gray", head_width=0.03)

    plt.arrow(df.loc[n, "X_SC_EME2000 [km]"]/1.e6, df.loc[n, "Y_SC_EME2000 [km]"] / 1.e6,           \
              ux_oa/3, uy_oa/3, color="green", head_width=0.03)
    
    
ax.set_xlabel("X - EME2000 [Gm]")
ax.set_ylabel("Y - EME2000 [Gm]")
plt.tight_layout()

plt.savefig("Plato_Moon_trajectory_xy.png", dpi=150)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

begin = 250
end = 650
middle = begin + (end-begin)/2

ax.scatter([0], [0], s=40, c="blue")
ax.scatter(df.loc[begin, "X_SC_EME2000 [km]"]/1.e6, df.loc[begin, "Z_SC_EME2000 [km]"] / 1.e6, s=40, c="red")


ax.plot(df.loc[begin:end, "X_MOON_EME2000 [km]"]/1.e6, df.loc[begin:end, "Z_MOON_EME2000 [km]"] / 1.e6, c="gray", linewidth=0.1)
ax.plot(df.loc[begin:end, "X_SC_EME2000 [km]"]/1.e6, df.loc[begin:end, "Z_SC_EME2000 [km]"] / 1.e6, c="red", linewidth=1)

for n in range(begin,end,30):

    plt.arrow(df.loc[n, "X_SC_EME2000 [km]"]/1.e6, df.loc[n, "Z_SC_EME2000 [km]"] / 1.e6,           \
              (df.loc[n, "X_SUN_EME2000 [km]"]/1.e6 - df.loc[n, "Z_SC_EME2000 [km]"]/1.e6)/1000,    \
              (df.loc[n, "Y_SUN_EME2000 [km]"]/1.e6 - df.loc[n, "Z_SC_EME2000 [km]"]/1.e6)/1000,    \
              color="orange", head_width=0.03)

    plt.arrow(df.loc[n, "X_SC_EME2000 [km]"]/1.e6, df.loc[n, "Z_SC_EME2000 [km]"] / 1.e6,           \
              (df.loc[n, "X_MOON_EME2000 [km]"]/1.e6 - df.loc[n, "Z_SC_EME2000 [km]"]/1.e6)/6,    \
              (df.loc[n, "Y_MOON_EME2000 [km]"]/1.e6 - df.loc[n, "Z_SC_EME2000 [km]"]/1.e6)/6,    \
              color="gray", head_width=0.03)

    plt.arrow(df.loc[n, "X_SC_EME2000 [km]"]/1.e6, df.loc[n, "Z_SC_EME2000 [km]"] / 1.e6,           \
              ux_oa/4, uz_oa/4, color="green", head_width=0.03)
    
    
ax.set_xlabel("X - EME2000 [Gm]")
ax.set_ylabel("Z - EME2000 [Gm]")
plt.tight_layout()

plt.savefig("Plato_Moon_trajectory_xz.png", dpi=150)

## How the moon is seen from Plato

In [ ]:
dist_moon_sc = 1.5e6                                                 # [km]
radius_moon = 1738.1                                                 # [km]

In [ ]:
e_sun = np.array([0.0, 0.0, 1.0])                                    # Direction of the Sun
e_oa = -e_sun                                                        # Direction of optical axis
r_c = dist_moon_sc * np.array([0.0, 1/np.sqrt(2), 1/np.sqrt(2)])     # Location of the spacecraft

In [ ]:
Ntheta = 200
Nphi = 400
integrand = []                   # Integrand of the visible parts
x = []
y = []
z = []
for theta in np.linspace(0.01, +np.pi, Ntheta):
    for phi in np.linspace(0.0, 2*np.pi, Nphi):
        cosalpha = np.cos(theta)
        e_perp = np.array([np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)])
        r_moon = radius_moon * e_perp
        e_c = (r_moon - r_c) / np.linalg.norm(r_moon - r_c)
        cosgamma = np.dot(e_perp, -e_c)
        cosbeta = np.dot(e_c, e_oa)
        
        if (cosbeta >= 0.0) and (cosalpha >= 0.0) and (cosgamma >= 0):
            integrand.append(cosalpha * cosbeta * cosgamma)
        else:
            integrand.append(0.0)
        x.append(np.sin(theta)*np.cos(phi))
        y.append(np.sin(theta)*np.sin(phi))
        z.append(np.cos(theta))

x = np.array(x)
y = np.array(y)
z = np.array(z)
integrand = np.array(integrand)

In [ ]:
my_df = pd.DataFrame({"x": x, "y": y, "z": z, "f": integrand})
my_df.head()

In [ ]:
fig = px.scatter_3d(my_df, x='x', y='y', z='z', color='f', size_max=2, opacity=1.0, \
                    labels={'x': 'X', 'y': 'Y', 'z': 'Z'},                       \
                    color_continuous_scale=px.colors.sequential.gray)

camera = dict(
    up=dict(x=-1, y=0, z=0),
    center=dict(x=0, y=0, z=0),
    eye=dict(x=0, y=1.5, z=1.5)
)

coloraxis_colorbar=dict(
    thicknessmode="pixels", thickness=20,
    lenmode="pixels", len=500
)

fig.update_layout(autosize=False, width=900, height=900,                 \
                  scene_camera=camera,                                   \
                  coloraxis_colorbar=coloraxis_colorbar)

#fig.write_image("MoonFromPlato.png", engine="kaleido")

fig.show()

## The Point Source Transmittance function 

The PST tables provided by INAF take into account:
* Coating
* Particulate contamination
* Transmission efficiency
* QE of the CCC.

In [ ]:
pst_az00 = pd.read_csv("../../../inputfiles/Plato_pst_az00.csv")
pst_az45 = pd.read_csv("../../../inputfiles/Plato_pst_az45.csv")
pst_az90 = pd.read_csv("../../../inputfiles/Plato_pst_az90.csv")
pst_az135 = pd.read_csv("../../../inputfiles/Plato_pst_az135.csv")
pst_az180 = pd.read_csv("../../../inputfiles/Plato_pst_az180.csv")

In [ ]:
pst_az00.head()

In [ ]:
Nazimuth = 5
Npolarangle = len(pst_az00)
polarangles = pst_az00['rho'].values
pst = np.zeros((Nazimuth, Npolarangle))
pst[0] = pst_az00['500nm']
pst[1,:len(pst_az45)] = pst_az45['500nm']
pst[2,:len(pst_az90)] = pst_az90['500nm']
pst[3,:len(pst_az135)] = pst_az135['500nm']
pst[4,:len(pst_az180)] = pst_az180['500nm']

In [ ]:
pst[0]

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

ax.plot(polarangles, pst[0], label=r"$\zeta$ = $0^{\circ}$")
ax.plot(polarangles, pst[1], label=r"$\zeta$ = $45^{\circ}$")
ax.plot(polarangles, pst[2], label=r"$\zeta$ = $90^{\circ}$")
ax.plot(polarangles, pst[3], label=r"$\zeta$ = $135^{\circ}$")
ax.plot(polarangles, pst[4], label=r"$\zeta$ = $180^{\circ}$")

ax.set_yscale("log", base=10)

ax.set_xlim(40,90)
ax.set_ylim(1e-8, 1e-3)
ax.set_xlabel(r"$\rho$ [deg]")
ax.set_ylabel(r"flux / mm$^2$ / (unit of incoming flux)")
ax.legend(loc="upper right")
ax.set_title(r"$\lambda=500$ nm")
plt.savefig("Plato_pst_500nm.png", dpi=150)
plt.show()

For an angle of $\rho=45^{\circ}$, we see PST values between $2\cdot 10^{-4}$ and $2\cdot 10^{-6}$ depending on the azimuthal angle.

In [ ]:
f_PST_min = 2.0e-4
f_PST_max = 2.0e-6

## Rough estimate of the number of electrons due to straylight with moon on 45 degrees

In [ ]:
Tsun = 5770.                                                           # [K]

In [ ]:
C = 299792458.0                                                        # [m/s]
H = 6.62607015e-34                                                     # [J/Hz]
kB = 1.380649e-23                                                      # [J/K]

def blackbody(T, lamda):
    return 2 * H * C**2 / lamda**5 / (np.exp(H*C/lamda/kB/T) - 1)      # J/s/m^2/m/sr 

In [ ]:
lambdaC = 550.0e-9                                                     # Central wavelength of Plato passband [m]
bandwidth = 532.0e-9                                                    # FWHM of Plato passband.              [m]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
lamda = np.linspace(200e-9, 1000e-9, 100)
ax.plot(lamda*1e9, blackbody(Tsun, lamda))
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Blackbody radiance [J/s/m2/m/sr]")
ax.set_yscale("log", base=10)
ax.set_xlim(lamda[0]*1e9, lamda[-1]*1e9)
plt.tight_layout()

In [ ]:
radiance, error = integrate.quad(lambda x: blackbody(Tsun, x), lambdaC-bandwidth/2, lambdaC+bandwidth/2)
print("Radiance: ", radiance, "J/s/m^2/sr")
energy_photon_lambdaC = H*C/lambdaC
photonflux = radiance / energy_photon_lambdaC
print("Photon flux: ", photonflux, "photons/s/m^2/sr")

In [ ]:
dist_moon_sc = 1.5e6                                                   # [km]
dist_moon_sun = 1.5e8                                                  # [km]
radius_moon = 1738.1                                                   # [km]
radius_sun = 695700.0                                                  # [km]

In [ ]:
albedo_moon = 0.12

In [ ]:
exposure_time = 25.0                                                   # [s]
light_collecting_area = 113.1e-4                                       # Of 1 camera [m^2]  
pixel_area = 18.0e-6 * 18.0e-6                                         # Of 1 pixel [m^2]

In [ ]:
e_sun = np.array([0.0, 0.0, 1.0])                                      # Direction of the Sun
e_oa = -e_sun                                                          # Direction of optical axis
r_c = dist_moon_sc * np.array([0.0, 1/np.sqrt(2), 1/np.sqrt(2)])       # Location of the spacecraft

In [ ]:
Ntheta = 200
Nphi = 400
thetas = np.linspace(0.01, +np.pi, Ntheta)
phis = np.linspace(0.0, 2*np.pi, Nphi)
DeltaTheta = thetas[1] - thetas[0]
DeltaPhi = phis[1] - phis[0]

flux = 0.0

for theta in thetas:
    for phi in phis:
        cosalpha = np.cos(theta)
        e_perp = np.array([np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)])
        r_moon = radius_moon * e_perp
        e_c = (r_moon - r_c) / np.linalg.norm(r_moon - r_c)
        cosgamma = np.dot(e_perp, -e_c)
        cosbeta = np.dot(e_c, e_oa)
        
        if (cosbeta >= 0.0) and (cosalpha >= 0.0) and (cosgamma >= 0):
            flux += cosalpha * cosbeta * cosgamma * np.sin(theta)
            
flux *= DeltaTheta * DeltaPhi * np.pi                                                       \
            * radius_sun**2 * radius_moon**2 / dist_moon_sc**2 / dist_moon_sun**2               \
            * albedo_moon                                                                       \
            * photonflux * exposure_time * light_collecting_area

irradiance = flux / light_collecting_area

print("Flux of the moon:", flux, "(log =", np.log10(flux), ") [ph/exposure]")
print("Entrance irradiance of the moon:", irradiance, "[ph/exposure/m^2] (log =", np.log10(irradiance), ")")

In [ ]:
print("Photons/exposure/pix reaching the sensor (low end baffle)", irradiance * pixel_area * f_PST_min)
print("Photons/exposure/pix reaching the sensor (high end baffle)", irradiance * pixel_area * f_PST_max)

## Time evolution of the lunar straylight for LOPS2

In [ ]:
radius_moon = 1738.1                                                   # [km]
radius_sun = 695700.0                                                  # [km]

radiance, error = integrate.quad(lambda x: blackbody(Tsun, x), lambdaC-bandwidth/2, lambdaC+bandwidth/2)
energy_photon_lambdaC = H*C/lambdaC
photonflux = radiance / energy_photon_lambdaC

exposure_time = 25.0                                                   # [s]
light_collecting_area = 113.1e-4                                       # Of 1 camera [m^2]  
pixel_area = 18.0e-6 * 18.0e-6                                         # Of 1 pixel [m^2]

albedo_moon = 0.12

In [ ]:
lops2 = SkyCoord('06h21m14.5s', '-47d53m13s', frame='icrs')
ux_oa = np.cos(lops2.dec.rad) * np.cos(lops2.ra.rad)                # Pointing direction of platform in EME2000 RF
uy_oa = np.cos(lops2.dec.rad) * np.sin(lops2.ra.rad)
uz_oa = np.sin(lops2.dec.rad)

e_sun = np.array([0.0, 0.0, 1.0])                                   # Direction of the Sun in Moon-centered RF

In [ ]:
Ntheta = 100
Nphi = 100
thetas = np.linspace(0.01, +np.pi, Ntheta)
phis = np.linspace(0.0, 2*np.pi, Nphi)
DeltaTheta = thetas[1] - thetas[0]
DeltaPhi = phis[1] - phis[0]

itime_begin = 200                                       # Skip the first 199 time points of the orbit file
itime_end   = 800                                       # We're only interested in the first two years

flux = np.zeros(itime_end - itime_begin, dtype=float)
max_cosbeta = np.zeros(itime_end - itime_begin, dtype=float)

for itime in range(itime_begin, itime_end):

    dist_moon_sc = df.loc[itime, "dist_sc_moon"]                                                   # [km]
    dist_moon_sun = df.loc[itime, "dist_sun_moon"] 

    # r_cam is the vector pointing from the Moon to the spacecraft
    # The following coordinates are in the EME2000 reference frame, and are in [km].
    
    r_cam_eme2000 = np.array([df.loc[itime, 'X_SC_EME2000 [km]'] - df.loc[itime, 'X_MOON_EME2000 [km]'],          \
                              df.loc[itime, 'Y_SC_EME2000 [km]'] - df.loc[itime, 'Y_MOON_EME2000 [km]'],          \
                              df.loc[itime, 'Z_SC_EME2000 [km]'] - df.loc[itime, 'Z_MOON_EME2000 [km]'] ])
    
    x_sun = df.loc[itime, 'X_SUN_EME2000 [km]'] - df.loc[itime, 'X_MOON_EME2000 [km]']
    y_sun = df.loc[itime, 'Y_SUN_EME2000 [km]'] - df.loc[itime, 'Y_MOON_EME2000 [km]']
    z_sun = df.loc[itime, 'Z_SUN_EME2000 [km]'] - df.loc[itime, 'Z_MOON_EME2000 [km]']
    
    epsilon = np.arctan2(x_sun, y_sun)
    eta = np.arctan2(np.sqrt(x_sun**2 + y_sun**2), z_sun)
    
    rot1 = np.array([[np.cos(epsilon), -np.sin(epsilon), 0.0],                       \
                     [np.sin(epsilon),  np.cos(epsilon), 0.0],                       \
                     [    0.0,              0.0,         1.0]])
    
    rot2 = np.array([[1.0,      0.0,           0.0   ],                              \
                     [0.0,  np.cos(eta), -np.sin(eta)],                              \
                     [0.0,  np.sin(eta),  np.cos(eta)]])
    
    e_oa = rot2 @ rot1 @ np.array([[ux_oa], [uy_oa], [uz_oa]])
    
    r_cam = rot2 @ rot1 @ r_cam_eme2000                 # Conversion to Moon-centric coordinates
    
    for theta in thetas:
        for phi in phis:
            e_perp = np.array([np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)])
            r_moon = radius_moon * e_perp
            e_cam  = (r_moon - r_cam) / np.linalg.norm(r_moon - r_cam)
            cosalpha = np.cos(theta)
            cosgamma = np.dot(e_perp, -e_cam)
            cosbeta  = np.dot(e_cam, e_oa)

            if (cosbeta >= 0.0) and (cosalpha >= 0.0) and (cosgamma >= 0):
                flux[itime - itime_begin] += cosalpha * cosbeta * cosgamma * np.sin(theta)
                if cosbeta > max_cosbeta[itime - itime_begin]:
                    max_cosbeta[itime - itime_begin] = cosbeta

    flux[itime - itime_begin] *= DeltaTheta * DeltaPhi * np.pi                                               \
                               * radius_sun**2 * radius_moon**2 / dist_moon_sc**2 / dist_moon_sun**2         \
                               * albedo_moon                                                                 \
                               * photonflux * exposure_time * light_collecting_area

    
irradiance = flux / light_collecting_area
scattered_moonlight_min = irradiance * pixel_area * f_PST_min
scattered_moonlight_max = irradiance * pixel_area * f_PST_max

In [ ]:
fig, ax = plt.subplots(4,1, sharex=True, figsize=(10, 17))
fig.subplots_adjust(hspace=0)

ax[0].plot(df.loc[itime_begin:itime_end-1, "jd"] - 2400000, irradiance)
ax[0].set_ylabel("Irradiance [ph/exp/m^2]")
ax[0].grid(c='gainsboro', which='both')

ax[1].plot(df.loc[itime_begin:itime_end-1, "jd"] - 2400000, 
           df.loc[itime_begin:itime_end-1, "dist_sc_moon"]**(-2) 
           * df.loc[itime_begin:itime_end-1, "dist_sun_moon"]**(-2))
ax[1].set_ylabel(r"$d_{Cam,Moon}^{-2} \cdot d_{Sun,Moon}^{-2}$ [km$^{-4}$]")
ax[1].set_yscale("log", base=10)
ax[1].grid(c='gainsboro', which='both')

ax[2].plot(df.loc[itime_begin:itime_end-1, "jd"] - 2400000, max_cosbeta)
ax[2].set_ylabel("Max(cos(beta))")
ax[2].grid(c='gainsboro', which='both')

ax[3].plot(df.loc[itime_begin:itime_end-1, "jd"] - 2400000, scattered_moonlight_min)
ax[3].set_ylabel("Scatter background [e-/exp/pix]")
ax[3].set_xlim(61230, 61750)
ax[3].xaxis.set_minor_locator(MultipleLocator(50))
ax[3].set_xlabel("Julian day - 2400000 [d]")
ax[3].grid(c='gainsboro', which='both')

plt.savefig("Plato_lops2.png", dpi=150)

## Scatter caused by bright infield objects

In [ ]:
dist = np.linspace(0, 80, 1000)                             # Distance from main projection on FP [mm] 

In [ ]:
def Escat(dist):
    log10E = 1.92 * np.exp(-0.44857545 * dist) - 3.4249736 + 3.17 * np.exp(-0.04111625 * dist)
    return 10**log10E

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dist, Escat(dist))
ax.set_xlabel(r"Distance $r$ from main projection on the FP [mm]")
ax.set_ylabel(r"$E_{scat}$ [W/mm$^2$]")
ax.set_yscale("log", base=10)
ax.set_xlim(0,80)
plt.grid(True, color='gainsboro')
plt.tight_layout()
plt.savefig("Infield_scattered_irradiance_500nm.png", dpi=150)

The integral of this function over its entire range:

In [ ]:
integral, error = integrate.quad(Escat, 0, 80)
print(integral, error)

In [ ]:
(1 - integral * pixel_area / light_collecting_area)

The scattered irradiance at $r=0$: 

In [ ]:
Escat(0)